# Rice seedlings detection and counting

This notebook use xxx_CNN model and compare with the classic CV baseline

# 1. Install, import packages and wandb setup

In [1]:
%pip install geoai-py wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 601.8/601.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.5/667.5 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.

In [2]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

import geoai
import geoai.train as gtrain
import rasterio
import geopandas as gpd
from shapely.geometry import Polygon
import geopandas as gpd
import rasterio

from pathlib import Path
import xml.etree.ElementTree as ET
import json
import shutil
import glob

import kagglehub
from kaggle_secrets import UserSecretsClient
import wandb

In [3]:
# Get secret key from Kaggle
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("WANDB_API_KEY")

# Đăng nhập tự động
wandb.login(key=wandb_api)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nguyentanphuc020505 (nguyentanphuc020505-ho-chi-minh-city-university-of-techn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
config = {
    "model_name": "maskrcnn_resnet50_fpn",
    "num_channels": 4,
    "num_classes": 2,
    "batch_size": 4,
    "num_epochs": 100,
    "learning_rate": 0.0001,
    "tile_size": 512,
    "stride": 256,
    "val_split": 0.2,
    "project": "rice_seedlings_detection_counting",
    "name": "RSDC-MaskRCNN"
}

wandb.init(
    project=config["project"],
    name=config["name"],
    config=config
)

wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260505_143136-2wd18ri5
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run RSDC-MaskRCNN
wandb: ⭐️ View project at https://wandb.ai/nguyentanphuc020505-ho-chi-minh-city-university-of-techn/rice_seedlings_detection_counting
wandb: 🚀 View run at https://wandb.ai/nguyentanphuc020505-ho-chi-minh-city-university-of-techn/rice_seedlings_detection_counting/runs/2wd18ri5


# 2. Set up the folder and file paths

In [5]:
DATA_DIR = Path("/kaggle/input/datasets/ngtanphuc020505/rice-seedlings-dataset/riceseedling_dataset")

ANNOTATION_DIR = DATA_DIR / "annotations"
RAW_DIR = DATA_DIR / "raw"

WORKING_DIR = Path("/kaggle/working")

OUT_DIR = WORKING_DIR / "output"
VECTOR_DIR = WORKING_DIR / "vectors"

VECTOR_DIR.mkdir(parents=True, exist_ok=True)

print("Raw images:", len(list(RAW_DIR.glob("*.tif"))))
print("XML labels:", len(list(ANNOTATION_DIR.glob("*.xml"))))

Raw images: 8
XML labels: 8


# 3. View the metadata of a raster
We need to get the basic information about what a raster might look like, what is Affine Transformation (so we can change the annotations from .xml to .geojson).

In [6]:
ANNO_2_PATH = RAW_DIR / "2.tif"

In [7]:
geoai.get_raster_info(ANNO_2_PATH)

{'driver': 'GTiff',
 'width': 1527,
 'height': 1527,
 'count': 4,
 'dtype': 'uint8',
 'crs': 'EPSG:3826',
 'transform': Affine(0.005239030778695925, 0.0, 218293.41517708264,
        0.0, -0.005239030779305829, 2658467.4298639484),
 'bounds': BoundingBox(left=218293.41517708264, bottom=2658459.4298639484, right=218301.4151770817, top=2658467.4298639484),
 'resolution': (0.01, 0.01),
 'nodata': None,
 'band_stats': [{'band': 1,
   'min': 33.0,
   'max': 249.0,
   'mean': 127.35750509600386,
   'std': 16.259917191192397},
  {'band': 2,
   'min': 36.0,
   'max': 254.0,
   'mean': 124.59662164857065,
   'std': 17.472386102087714},
  {'band': 3,
   'min': 46.0,
   'max': 230.0,
   'mean': 122.81249064535373,
   'std': 13.55456635998707},
  {'band': 4, 'min': 255.0, 'max': 255.0, 'mean': 255.0, 'std': 0.0}]}

The raster above has width x height x bands = 1527 x 1527 x 4. And the \
'transform': Affine(0.005239030778695925, 0.0, 218293.41517708264, 0.0, -0.005239030779305829, 2658467.4298639484) 

Affine matrix has 6 key parameters: [a, b, c, d, e, f]. Based on your data, these are:
- a: $0.005239$ (Pixel width)
- b: $0.0$ (Row rotation/skew)
- c: $218293.415$ (Top-Left X coordinate in meters)
- d: $0.0$ (Column rotation/skew)
- e: $-0.005239$ (Pixel height - negative because images draw top-to-bottom, but maps draw bottom-to-top)
- f: $2658467.429$ (Top-Left Y coordinate in meters)

Also, the top-left corner (c and f) of the image is:
- X (Easting) = 218293.415
- Y (Northing) = 2658467.429

Use the informations above to convert the vectors from .xml to .geojson

# 4. Change the .xml annotation to .geojson for geoai

We're going to use geoai to train the model. This requires the annotations to be in .geojson format. However, the annotations right now are in .xml format

This function reads a .xml file and returns a list of dictionaries. Each dictionary is an object containing information of a bbox: xmin, ymin, xmax, ymax

In [8]:
def parse_xml(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    ret = []

    for bbox in root.iter("bndbox"):
        info = {}
        info["xmin"] = int(bbox.find("xmin").text)
        info["ymin"] = int(bbox.find("ymin").text)
        info["xmax"] = int(bbox.find("xmax").text)
        info["ymax"] = int(bbox.find("ymax").text)

        ret.append(info)

    return ret

To convert any pixel coordinate $(x, y)$ to a geographic coordinate $(X, Y)$, the system applies this linear algebra formula:
- $X = (a \cdot x) + (b \cdot y) + c$
- $Y = (d \cdot x) + (e \cdot y) + f$

But with rasterio, we can multiply the Affine object with the coordinate $(x, y)$ and get the same result

Convert the .xml files to .geojson

In [9]:
for xml in ANNOTATION_DIR.iterdir():
    xml_path = str(xml.resolve())
    if ".txt" in xml_path:
        continue

    xml_name = xml.name[0]
    raster_path = RAW_DIR / f"{xml_name}.tif"

    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        t = src.transform

    polygons = []
    for bbox in parse_xml(xml_path):
        xmin, ymin = bbox["xmin"], bbox["ymin"]
        xmax, ymax = bbox["xmax"], bbox["ymax"]

        A = t * (xmin, ymin)  # top-left
        B = t * (xmax, ymin)  # top-right
        C = t * (xmax, ymax)  # bottom-right
        D = t * (xmin, ymax)  # bottom-left

        polygons.append(Polygon([A, B, C, D, A]))

    gdf = gpd.GeoDataFrame(
        {"class": ["riceseedling"] * len(polygons)},
        geometry=polygons,
        crs=raster_crs
    )
    out_path = VECTOR_DIR / f"{xml_name}.geojson"
    gdf.to_file(out_path, driver="GeoJSON")
    print(f"{xml_name}: {len(polygons)} annotations, CRS={raster_crs}")

2: 1000 annotations, CRS=EPSG:3826
8: 1005 annotations, CRS=EPSG:3826
6: 1002 annotations, CRS=EPSG:3826
7: 1029 annotations, CRS=EPSG:3826
1: 898 annotations, CRS=EPSG:3826
4: 985 annotations, CRS=EPSG:3826
3: 1019 annotations, CRS=EPSG:3826
5: 996 annotations, CRS=EPSG:3826


# 5. Train the object detection model

## Create train/val data from images 1-6 and test data from images 7, 8

We have to chip down the image because it is too large to feed the model

In [10]:
# Create the temp directories
for raster_path in sorted(RAW_DIR.glob("*.tif")):
    name = raster_path.stem
    vector_path = VECTOR_DIR / f"{name}.geojson"

    print(f"Chipping:\n + raster {raster_path}\n + vector {vector_path}")
    geoai.export_geotiff_tiles(
        in_raster=raster_path,
        out_folder=OUT_DIR / name,
        in_class_data=vector_path,
        tile_size=512,
        stride=256,
        buffer_radius=0,
    )


Chipping:
 + raster /kaggle/input/datasets/ngtanphuc020505/rice-seedlings-dataset/riceseedling_dataset/raw/1.tif
 + vector /kaggle/working/vectors/1.geojson


Generated: 25, With features: 25: 100%|██████████| 25/25 [00:05<00:00,  4.35it/s]


Chipping:
 + raster /kaggle/input/datasets/ngtanphuc020505/rice-seedlings-dataset/riceseedling_dataset/raw/2.tif
 + vector /kaggle/working/vectors/2.geojson


Generated: 25, With features: 25: 100%|██████████| 25/25 [00:06<00:00,  4.14it/s]


Chipping:
 + raster /kaggle/input/datasets/ngtanphuc020505/rice-seedlings-dataset/riceseedling_dataset/raw/3.tif
 + vector /kaggle/working/vectors/3.geojson


...

Generated: 25, With features: 25: 100%|██████████| 25/25 [00:06<00:00,  4.12it/s]


Chipping:
 + raster /kaggle/input/datasets/ngtanphuc020505/rice-seedlings-dataset/riceseedling_dataset/raw/7.tif
 + vector /kaggle/working/vectors/7.geojson


Generated: 25, With features: 25: 100%|██████████| 25/25 [00:06<00:00,  4.11it/s]


Chipping:
 + raster /kaggle/input/datasets/ngtanphuc020505/rice-seedlings-dataset/riceseedling_dataset/raw/8.tif
 + vector /kaggle/working/vectors/8.geojson


Generated: 25, With features: 25: 100%|██████████| 25/25 [00:06<00:00,  3.90it/s]


Create train/val data from images 1-6

In [11]:
TRAIN_IMG_DIR = OUT_DIR / "train" / "images"
TRAIN_IMG_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_LABEL_DIR = OUT_DIR / "train" / "labels"
TRAIN_LABEL_DIR.mkdir(parents=True, exist_ok=True)

count = 0
for i in range(1, 7):

    OLD_IMG_DIR = OUT_DIR / f"{i}" / "images"
    OLD_LABEL_DIR = OUT_DIR / f"{i}" / "labels"
    for img, label in zip(OLD_IMG_DIR.iterdir(), OLD_LABEL_DIR.iterdir()):
        new_name = f"tile_{"0"*(6-len(str(count)))}{count}.tif"
        shutil.copy(str(img.resolve()), TRAIN_IMG_DIR / new_name)  # copy image
        shutil.copy(str(label.resolve()), TRAIN_LABEL_DIR / new_name)  # copy label
        count += 1

    shutil.rmtree(OUT_DIR / f"{i}")  # delete the temp directory

Create test data from images 7, 8

In [12]:
TEST_IMG_DIR = OUT_DIR / "test" / "images"
TEST_IMG_DIR.mkdir(parents=True, exist_ok=True)

TEST_LABEL_DIR = OUT_DIR / "test" / "labels"
TEST_LABEL_DIR.mkdir(parents=True, exist_ok=True)

count = 0
for i in range(7, 9):

    OLD_IMG_DIR = OUT_DIR / f"{i}" / "images"
    OLD_LABEL_DIR = OUT_DIR / f"{i}" / "labels"
    for img, label in zip(OLD_IMG_DIR.iterdir(), OLD_LABEL_DIR.iterdir()):
        new_name = f"tile_{"0"*(6-len(str(count)))}{count}.tif"
        shutil.copy(str(img.resolve()), TEST_IMG_DIR / new_name)  # copy image
        shutil.copy(str(label.resolve()), TEST_LABEL_DIR / new_name)  # copy label
        count += 1

    shutil.rmtree(OUT_DIR / f"{i}")  # delete the temp directory

## Train object detection model

In [13]:
current_epoch = 0

def patched_train_one_epoch(
    model, optimizer, data_loader, device, epoch, print_freq, scaler=None,
    orig=gtrain.train_one_epoch
):
    global current_epoch
    train_loss = orig(model, optimizer, data_loader, device, epoch, print_freq, scaler)
    current_epoch = epoch + 1
    
    # Log both scalar and full dict if possible
    wandb.log({
        "epoch": epoch, 
        "train/loss": float(train_loss)
    }, step=epoch)
    print(f"💩💩💩💩💩 Epoch {current_epoch} 💩💩💩💩💩")
    print(f"📊 W&B train loss: {float(train_loss):.4f}")
    return train_loss

def patched_evaluate(
    model, data_loader, device,
    orig=gtrain.evaluate,
    **kwargs
):
    global current_epoch
    
    # Call original evaluate
    metrics = orig(model, data_loader, device, **kwargs)
    
    log_dict = {
        "epoch": current_epoch,
        "val/loss": metrics.get("loss", 0),
        "val/IoU": metrics.get("IoU", 0),
    }
    
    # Add full COCO metrics
    try:
        coco_metrics = gtrain.evaluate_coco_metrics(
            model, data_loader, device, verbose=False
        )
        log_dict.update({
            f"val/{k}": v for k, v in coco_metrics.items()
        })
        print(f"📊 COCO mAP@0.5: {coco_metrics.get('mAP@0.5', 0):.4f} | "
              f"mAP@0.5:0.95: {coco_metrics.get('mAP@[0.5:0.95]', 0):.4f}")
    except Exception as e:
        print(f"⚠️ COCO metrics failed: {e}")
    
    wandb.log(log_dict, step=current_epoch)
    return metrics

gtrain.train_one_epoch = patched_train_one_epoch
gtrain.evaluate = patched_evaluate
print("✅ W&B patches applied")

✅ W&B patches applied


In [14]:
print(f"===== Total number of epochs: {config["num_epochs"]} =====")

===== Total number of epochs: 100 =====


In [15]:
geoai.train_MaskRCNN_model(
    model_name=config["model_name"],
    images_dir=f"{OUT_DIR}/train/images",
    labels_dir=f"{OUT_DIR}/train/labels",
    output_dir=f"{OUT_DIR}/models",
    num_channels=config["num_channels"],
    pretrained=True,
    batch_size=config["batch_size"],
    num_epochs=config["num_epochs"],
    learning_rate=config["learning_rate"],
    val_split=config["val_split"],
)

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 211MB/s]


💩💩💩💩💩 Epoch 1 💩💩💩💩💩
📊 W&B train loss: 5.1520
📊 COCO mAP@0.5: 0.0164 | mAP@0.5:0.95: 0.0049
💩💩💩💩💩 Epoch 2 💩💩💩💩💩
📊 W&B train loss: 3.1528
📊 COCO mAP@0.5: 0.0363 | mAP@0.5:0.95: 0.0085
💩💩💩💩💩 Epoch 3 💩💩💩💩💩
📊 W&B train loss: 2.4915
📊 COCO mAP@0.5: 0.0792 | mAP@0.5:0.95: 0.0184
💩💩💩💩💩 Epoch 4 💩💩💩💩💩
📊 W&B train loss: 2.1728
📊 COCO mAP@0.5: 0.1317 | mAP@0.5:0.95: 0.0329
💩💩💩💩💩 Epoch 5 💩💩💩💩💩
📊 W&B train loss: 1.9895
📊 COCO mAP@0.5: 0.1753 | mAP@0.5:0.95: 0.0440
💩💩💩💩💩 Epoch 6 💩💩💩💩💩
📊 W&B train loss: 1.8860
📊 COCO mAP@0.5: 0.2221 | mAP@0.5:0.95: 0.0544
💩💩💩💩💩 Epoch 7 💩💩💩💩💩
📊 W&B train loss: 1.8104
📊 COCO mAP@0.5: 0.2577 | mAP@0.5:0.95: 0.0629
💩💩💩💩💩 Epoch 8 💩💩💩💩💩
📊 W&B train loss: 1.7579
📊 COCO mAP@0.5: 0.2909 | mAP@0.5:0.95: 0.0722
💩💩💩💩💩 Epoch 9 💩💩💩💩💩
📊 W&B train loss: 1.6998
📊 COCO mAP@0.5: 0.3101 | mAP@0.5:0.95: 0.0793
💩💩💩💩💩 Epoch 10 💩💩💩💩💩
📊 W&B train loss: 1.6451
📊 COCO mAP@0.5: 0.3251 | mAP@0.5:0.95: 0.0839
💩💩💩💩💩 Epoch 11 💩💩💩💩💩
📊 W&B train loss: 1.5994
📊 COCO mAP@0.5: 0.3376 | mAP@0.5:0.95: 0.08

Log the best model artifact after training completes

In [16]:
best_model_path = OUT_DIR / "models" / "best_model.pth"
if best_model_path.exists():
    artifact = wandb.Artifact(
        name=config["name"],
        type="model",
        metadata={"epochs": config["num_epochs"], "learning_rate": config["learning_rate"]}
    )
    artifact.add_file(str(best_model_path), name="best_model.pth")
    wandb.run.log_artifact(artifact)
    print("✅ best_model.pth logged to W&B")


✅ best_model.pth logged to W&B


# 6. Evaluate the model

Load the best model

In [17]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
model_path = OUT_DIR / "models/best_model.pth"

# Define the model architecture
model = gtrain.get_detection_model(
    model_name=config["model_name"],
    num_classes=2,
    num_channels=config["num_channels"],
    pretrained=False
)

# Load the trained weights
state_dict = torch.load(model_path, map_location=device)
model.load_state_dict(state_dict)

model.to(device)
model.eval()

print("✅ Best Model loaded successfully")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 151MB/s]


✅ Best Model loaded successfully


In [18]:
image_paths = sorted(glob.glob(str(TEST_IMG_DIR / "*.tif")))
label_paths = sorted(glob.glob(str(TEST_LABEL_DIR / "*.tif")))

test_dataset = gtrain.ObjectDetectionDataset(
    image_paths=image_paths,
    label_paths=label_paths,
    transforms=gtrain.get_transform(train=False),
    num_channels=config["num_channels"]
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=gtrain.collate_fn,
    num_workers=2
)

print(f"✅ Test loader created with {len(test_dataset)} images")

✅ Test loader created with 50 images


Evaluate the model

In [19]:
basic_metrics = gtrain.evaluate(
    model=model,
    data_loader=test_loader,
    device=device,
    use_mask_iou=True  # true for Mask R-CNN
)

📊 COCO mAP@0.5: 0.2950 | mAP@0.5:0.95: 0.0855


In [20]:
coco_metrics = gtrain.evaluate_coco_metrics(
    model=model,
    data_loader=test_loader,
    device=device,
    class_names=["riceseedling"],
    verbose=True
)

100%|██████████| 13/13 [00:10<00:00,  1.26it/s]


Calculate R2, RSME, MAPE

In [21]:
def evaluate_counting(model, data_loader, device):
    preds_count = []
    gts_count = []
    
    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        
        for out, tgt in zip(outputs, targets):
            pred_count = len(out["boxes"])
            gt_count = len(tgt["boxes"])
            preds_count.append(pred_count)
            gts_count.append(gt_count)
    
    r2 = r2_score(gts_count, preds_count)
    rmse = np.sqrt(mean_absolute_error(gts_count, preds_count))
    mape = np.mean(np.abs((np.array(gts_count) - np.array(preds_count)) / np.array(gts_count))) * 100

    return (r2, rmse, mape)
    

In [22]:
print("\n" + "="*40)
print("FINAL TEST RESULTS")
print("="*40)
print(f"IoU: {basic_metrics.get('IoU', 0):.4f}")
for k, v in coco_metrics.items():
    print(f"{k:15}: {v:.4f}")

r2, rmse, mape = evaluate_counting(model, test_loader, device)
print(f"Counting -> R²: {r2:.4f} | RMSE: {rmse:.2f} | MAPE: {mape:.2f}%")


FINAL TEST RESULTS
IoU: 0.4985
AP@0.5/riceseedling: 0.2950
mAP@0.5        : 0.2950
mAP@0.75       : 0.0220
mAP@[0.5:0.95] : 0.0855
Counting -> R²: -7.3400 | RMSE: 4.38 | MAPE: 15.79%


## Finish the wandb run

In [23]:
wandb.finish()

wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:              epoch ▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇███
wandb:         train/loss █▆▄▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:            val/IoU ▁▅▆▇▇▇▇█████████████████████████████████
wandb:           val/loss █▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:        val/mAP@0.5 ▁▄▄▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇█████████████████████▆
wandb:       val/mAP@0.75 ▁▁▂▂▂▃▄▄▄▄▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██████████████▄
wandb: val/mAP@[0.5:0.95] ▁▂▃▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇██████████████████
wandb: 
wandb: Run summary:
wandb:              epoch 100
wandb:         train/loss 1.27968
wandb:            val/IoU 0.49845
wandb:           val/loss 1.37318
wandb:        val/mAP@0.5 0.29502
wandb:       val/mAP@0.75 0.02205
wandb: val/mAP@[0.5:0.95] 0.08546
wandb: 
wandb: 🚀 View run RSDC-MaskRCNN at: https://wandb.ai/nguyentanphuc020505-ho-chi-minh-city-university-of-techn/rice_seedlings_detection_counting/runs/2wd18ri5
wandb: ⭐️ View project at: https://wandb.ai/ng